In [1]:
import chromadb
from sentence_transformers import SentenceTransformer

In [2]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
client = chromadb.PersistentClient(
    path="../vector_db"
)

In [4]:
collection = client.create_collection(
    name="scam_profiles"
)

In [5]:
scam_profiles = [
    "Pay registration fee for interview",
    "Guaranteed crypto investment returns",
    "Urgent bank account verification required",
    "Send OTP to confirm lottery winnings",
    "Official HR account hiring immediately",
    "Investment doubling scheme"
]

In [6]:
embeddings = embedding_model.encode(
    scam_profiles
).tolist()

In [7]:
collection.add(
    documents=scam_profiles,
    embeddings=embeddings,
    ids=[f"id{i}" for i in range(len(scam_profiles))]
)

In [8]:
query = "Pay onboarding fee to secure your interview"

In [9]:
query_embedding = embedding_model.encode(
    [query]
).tolist()

In [10]:
results = collection.query(
    query_embeddings=query_embedding,
    n_results=3
)

In [11]:
print(results['documents'])

[['Pay registration fee for interview', 'Official HR account hiring immediately', 'Urgent bank account verification required']]


In [12]:
prompt = f"""
Profile Analysis:

Query:
{query}

Retrieved Scam Patterns:
{results['documents'][0]}

Explain why this profile may be suspicious.
"""

print(prompt)


Profile Analysis:

Query:
Pay onboarding fee to secure your interview

Retrieved Scam Patterns:
['Pay registration fee for interview', 'Official HR account hiring immediately', 'Urgent bank account verification required']

Explain why this profile may be suspicious.



In [13]:
from openai import OpenAI

In [15]:
explanation = f"""
This profile appears suspicious because it resembles known scam patterns.

Reasons:
- Uses payment-related language
- Similar to recruiter scam profiles
- Contains urgency-based wording
- Matches previously stored scam examples

Retrieved Scam Patterns:
{results['documents'][0]}
"""

print(explanation)


This profile appears suspicious because it resembles known scam patterns.

Reasons:
- Uses payment-related language
- Similar to recruiter scam profiles
- Contains urgency-based wording
- Matches previously stored scam examples

Retrieved Scam Patterns:
['Pay registration fee for interview', 'Official HR account hiring immediately', 'Urgent bank account verification required']



The system supports LLM integration for AI-generated investigation reports. Due to API quota constraints during development, a mock reasoning layer was used to simulate investigation explanations